In [12]:
# SIFT-Anchored RST-Robust Watermarking (Simple Demo)
# - Color host (BGR). Embed on Y (luma) channel only.
# - 32x32 binary watermark (auto-resize + binarize if needed).
# - SIFT selects keypoints; each keypoint => a rotation/scale-normalized 64x64 patch.
# - Embed 1 bit/patch via DCT-QIM at a chosen mid-band coefficient.
# - Blind extraction: re-detect SIFT, same patch normalization, read bits, majority vote.

import cv2
import numpy as np
from pathlib import Path

# ---------------------------
# Utils: PSNR / BER
# ---------------------------
def psnr(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(np.float64); b = b.astype(np.float64)
    mse = np.mean((a - b) ** 2)
    if mse <= 1e-12: return 99.0
    return 10.0 * np.log10((255.0 ** 2) / mse)

def ber(bits_true: np.ndarray, bits_pred: np.ndarray) -> float:
    bits_true = bits_true.astype(np.uint8).ravel()
    bits_pred = bits_pred.astype(np.uint8).ravel()
    return np.mean(bits_true ^ bits_pred)

# ---------------------------
# Watermark prep
# ---------------------------
def load_binary_wm(path: str, size: int = 32) -> np.ndarray:
    wm = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if wm is None:
        raise FileNotFoundError(path)
    wm = cv2.resize(wm, (size, size), interpolation=cv2.INTER_AREA)
    # binarize
    _, wm_bin = cv2.threshold(wm, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return wm_bin.astype(np.uint8)

# ---------------------------
# SIFT & patch normalization
# ---------------------------
def sift_keypoints(gray: np.ndarray, nfeatures: int = 4000, contrast=0.03):
    sift = cv2.SIFT_create(nfeatures=nfeatures, contrastThreshold=contrast, edgeThreshold=10, sigma=1.6)
    kps = sift.detect(gray, None)
    # sort: strongest first
    kps = list(kps)
    kps.sort(key=lambda k: k.response, reverse=True)
    return kps

def affine_from_keypoint(kp: cv2.KeyPoint, patch_size: int = 64) -> tuple[np.ndarray, np.ndarray]:
    """
    Build 2x3 affine matrices:
      M     : maps source image -> normalized patch (patch_size x patch_size)
      M_inv : maps normalized patch -> source image
    We align rotation to 0 and scale so that kp.size becomes patch_size.
    """
    x0, y0 = kp.pt
    theta = -kp.angle * np.pi / 180.0  # rotate source by -angle to align orientation
    s = patch_size / (kp.size + 1e-6)  # scale so kp.size -> patch_size
    c, s1 = np.cos(theta), np.sin(theta)

    # A = s * R(-theta)
    A = np.array([[ s*c,  s*s1],
                  [-s*s1, s*c ]], dtype=np.float32)
    # t so that (x0,y0) -> (patch_center)
    pc = np.array([patch_size/2.0, patch_size/2.0], dtype=np.float32)
    p  = np.array([x0, y0], dtype=np.float32)
    t  = pc - A @ p
    M  = np.hstack([A, t.reshape(2,1)])

    # Inverse: x = A^{-1}(y - t)
    det = A[0,0]*A[1,1] - A[0,1]*A[1,0]
    A_inv = (1.0/det) * np.array([[ A[1,1], -A[0,1]],
                                  [-A[1,0],  A[0,0]]], dtype=np.float32)
    t_inv = -A_inv @ t
    M_inv = np.hstack([A_inv, t_inv.reshape(2,1)])
    return M, M_inv

def extract_patch(gray: np.ndarray, M: np.ndarray, patch_size: int = 64) -> np.ndarray:
    return cv2.warpAffine(gray, M, (patch_size, patch_size),
                          flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT101)

# ---------------------------
# QIM on one DCT coefficient
# ---------------------------
def embed_bit_qim_dct(patch: np.ndarray, bit: int, delta: float = 18.0,
                      uv: tuple[int,int]=(10,10)) -> np.ndarray:
    f = patch.astype(np.float32)
    F = cv2.dct(f)
    u, v = uv
    c = F[u, v]
    q = np.round(c / delta)
    if (q % 2) != bit:
        # flip to nearest opposite parity with minimal change
        # choose direction that yields smaller |c - delta*(q±1)|
        up_err   = abs(c - delta*(q+1))
        down_err = abs(c - delta*(q-1))
        q = q + 1 if up_err < down_err else q - 1
    F[u, v] = q * delta
    g = cv2.idct(F)
    return np.clip(g, 0, 255).astype(np.float32)

def extract_bit_qim_dct(patch: np.ndarray, delta: float = 18.0,
                        uv: tuple[int,int]=(10,10)) -> int:
    f = patch.astype(np.float32)
    F = cv2.dct(f)
    u, v = uv
    q = int(np.round(F[u, v] / delta))
    return int(q & 1)

# ---------------------------
# Assignment of bits to keypoints (repeat for robustness)
# ---------------------------
def assign_pairs(n_kp: int, n_bits: int, repeats: int, seed: int = 1234):
    rng = np.random.default_rng(seed)
    order = np.arange(n_kp)
    rng.shuffle(order)
    max_pairs = min(n_kp, n_bits * repeats)
    order = order[:max_pairs]

    # Distribute keypoints round-robin over bits to keep repetition balanced
    pairs = []
    for i, kp_idx in enumerate(order):
        bit_idx = i % n_bits
        pairs.append((kp_idx, bit_idx))
    return pairs

# ---------------------------
# Embed / Extract
# ---------------------------
def embed_watermark(img_bgr: np.ndarray, wm_bits_2d: np.ndarray,
                    repeats: int = 3, delta: float = 18.0, uv=(10,10),
                    patch_size: int = 64, seed: int = 1234):
    # Y channel
    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    Y, Cr, Cb = cv2.split(ycrcb)
    Yf = Y.astype(np.float32)

    kps = sift_keypoints(Y)
    n_bits = wm_bits_2d.size
    pairs = assign_pairs(len(kps), n_bits, repeats, seed=seed)
    if len(pairs) < n_bits:
        print(f"[warn] Only {len(pairs)} keypoints for {n_bits} bits. Fewer repetitions or fewer bits will be embedded.")

    # Precompute bit list
    wm_bits = wm_bits_2d.flatten().astype(np.uint8)

    for kp_idx, bit_idx in pairs:
        kp = kps[kp_idx]
        M, M_inv = affine_from_keypoint(kp, patch_size=patch_size)
        patch = extract_patch(Yf, M, patch_size=patch_size)
        mod   = embed_bit_qim_dct(patch, int(wm_bits[bit_idx]), delta=delta, uv=uv)

        # Accumulate only the difference, then warp back once
        diff = (mod - patch).astype(np.float32)
        diff_back = cv2.warpAffine(diff, M_inv, (Yf.shape[1], Yf.shape[0]),
                                   flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
        Yf += diff_back

    Y_new = np.clip(Yf, 0, 255).astype(np.uint8)
    out = cv2.merge([Y_new, Cr, Cb])
    out_bgr = cv2.cvtColor(out, cv2.COLOR_YCrCb2BGR)
    return out_bgr, len(pairs)

def extract_watermark(img_bgr: np.ndarray, n_side: int = 32,
                      repeats: int = 3, delta: float = 18.0, uv=(10,10),
                      patch_size: int = 64, seed: int = 1234) -> np.ndarray:
    ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    Y = ycrcb[:,:,0].astype(np.float32)

    kps = sift_keypoints(Y.astype(np.uint8))
    n_bits = n_side * n_side
    pairs = assign_pairs(len(kps), n_bits, repeats, seed=seed)

    # majority vote
    votes0 = np.zeros(n_bits, dtype=np.int32)
    votes1 = np.zeros(n_bits, dtype=np.int32)

    for kp_idx, bit_idx in pairs:
        kp = kps[kp_idx]
        M, _ = affine_from_keypoint(kp, patch_size=patch_size)
        patch = extract_patch(Y, M, patch_size=patch_size)
        b = extract_bit_qim_dct(patch, delta=delta, uv=uv)
        if b == 1: votes1[bit_idx] += 1
        else:      votes0[bit_idx] += 1

    wm_hat = (votes1 >= votes0).astype(np.uint8).reshape(n_side, n_side)
    return wm_hat

# ---------------------------
# Demo
# ---------------------------
if __name__ == "__main__":
    host_path = "images/cat.webp"
    wm_path   = "images/cat-sm.jpg"

    img = cv2.imread(host_path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(host_path)

    wm_bits = load_binary_wm(wm_path, size=32)

    cv2.imshow("wm", (wm_bits*255).astype(np.uint8))
    cv2.waitKey(0)

    # Embed
    wm_img, used = embed_watermark(img, wm_bits, repeats=3, delta=18.0, uv=(10,10), patch_size=64, seed=1234)
    cv2.imwrite("watermarked.png", wm_img)
    print(f"Embedded using {used} SIFT keypoints.")

    # Evaluate (optional)
    print("PSNR (host vs watermarked):", psnr(img, wm_img))

    # Blind extract
    wm_hat = extract_watermark(wm_img, n_side=32, repeats=3, delta=18.0, uv=(10,10), patch_size=64, seed=1234)
    cv2.imwrite("wm_extracted.png", (wm_hat*255).astype(np.uint8))

    # If you know the ground truth, compute BER:
    print("Clean BER:", ber(wm_bits, wm_hat))

Embedded using 3072 SIFT keypoints.
PSNR (host vs watermarked): 51.57671454422108
Clean BER: 0.85546875


In [8]:
def load_binary_wm(path: str, size: int = 32) -> np.ndarray:
    wm = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if wm is None:
        raise FileNotFoundError(path)
    wm = cv2.resize(wm, (size, size), interpolation=cv2.INTER_AREA)
    # binarize
    _, wm_bin = cv2.threshold(wm, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return wm_bin.astype(np.uint8)

host_path = "images/cat.webp"
wm_path   = "images/cat-sm.jpg"

img = cv2.imread(host_path, cv2.IMREAD_COLOR)
if img is None:
    raise FileNotFoundError(host_path)

wm_bits = load_binary_wm(wm_path, size=32)

ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)
Yf = Y.astype(np.float32)


sift = cv2.SIFT_create(nfeatures=4000, contrastThreshold=0.03, edgeThreshold=10, sigma=1.6)
kps, descriptors = sift.detectAndCompute(Y, None)

kps = list(kps)  # Convert to list if needed
kps.sort(key=lambda k: k.response, reverse=True)
kps

[< cv2.KeyPoint 00000268C96E9320>,
 < cv2.KeyPoint 00000268C96E9350>,
 < cv2.KeyPoint 00000268C9055260>,
 < cv2.KeyPoint 00000268C9055290>,
 < cv2.KeyPoint 00000268C90552C0>,
 < cv2.KeyPoint 00000268C9056520>,
 < cv2.KeyPoint 00000268C96172A0>,
 < cv2.KeyPoint 00000268C905BAB0>,
 < cv2.KeyPoint 00000268C905BB40>,
 < cv2.KeyPoint 00000268C907D290>,
 < cv2.KeyPoint 00000268C9057060>,
 < cv2.KeyPoint 00000268C96E8C00>,
 < cv2.KeyPoint 00000268C9064060>,
 < cv2.KeyPoint 00000268C96EA8B0>,
 < cv2.KeyPoint 00000268C96E82A0>,
 < cv2.KeyPoint 00000268C96E82D0>,
 < cv2.KeyPoint 00000268C9065C80>,
 < cv2.KeyPoint 00000268C9065CB0>,
 < cv2.KeyPoint 00000268C90592F0>,
 < cv2.KeyPoint 00000268C96E97D0>,
 < cv2.KeyPoint 00000268C90654D0>,
 < cv2.KeyPoint 00000268C96177B0>,
 < cv2.KeyPoint 00000268C907F7E0>,
 < cv2.KeyPoint 00000268C9646130>,
 < cv2.KeyPoint 00000268C9646160>,
 < cv2.KeyPoint 00000268C9073EA0>,
 < cv2.KeyPoint 00000268C96179C0>,
 < cv2.KeyPoint 00000268C90556E0>,
 < cv2.KeyPoint 0000